In [ ]:
import sys
print(sys.executable)

In [ ]:
from pathlib import Path
import importlib
import csv

import pandas as pd


def _find_repo_root() -> Path:
    """Resolve repo root so data/ paths work regardless of notebook CWD."""
    start = Path.cwd().resolve()
    for d in [start, *start.parents]:
        if (d / "server" / "pyproject.toml").is_file():
            return d
    return start


def _resolve_rpc_reader_ctor():
    """Return a callable constructor from rpc_reader package variants."""
    rr = importlib.import_module("rpc_reader")

    direct_names = ("ReadRPC", "RPCReader")
    for name in direct_names:
        ctor = getattr(rr, name, None)
        if callable(ctor):
            return ctor

    for submodule in ("rpc_reader", "reader", "core"):
        try:
            mod = importlib.import_module(f"rpc_reader.{submodule}")
        except Exception:
            continue
        for name in direct_names:
            ctor = getattr(mod, name, None)
            if callable(ctor):
                return ctor

    visible = [name for name in dir(rr) if not name.startswith("_")]
    raise AttributeError(
        "Installed 'rpc_reader' module has no supported reader class. "
        f"module={getattr(rr, '__file__', '<unknown>')} attrs={visible[:25]}"
    )


def _to_header_dict(obj):
    """Normalize header metadata into a dict without dropping keys."""
    if obj is None:
        return {}
    if isinstance(obj, dict):
        return {str(k): v for k, v in obj.items()}
    if isinstance(obj, list):
        out = {}
        for i, item in enumerate(obj):
            out[f"item_{i}"] = item
        return out
    return {"raw_header": obj}


def _extract_channel_names(channels, width):
    names = []
    for i in range(width):
        ch = channels[i] if i < len(channels) else None

        name = None
        if isinstance(ch, str):
            name = ch
        elif isinstance(ch, dict):
            name = ch.get("name") or ch.get("Name") or ch.get("channel_name")
        elif ch is not None:
            name = (
                getattr(ch, "name", None)
                or getattr(ch, "Name", None)
                or getattr(ch, "channel_name", None)
            )

        names.append(str(name).strip() if name else f"channel_{i + 1}")

    # Ensure headers are unique and preserve full width.
    seen = {}
    unique = []
    for n in names:
        count = seen.get(n, 0)
        seen[n] = count + 1
        unique.append(n if count == 0 else f"{n}_{count + 1}")
    return unique


def _load_via_rpc_reader(rsp_path: Path):
    ctor = _resolve_rpc_reader_ctor()
    reader = ctor(rsp_path)

    for load_method in ("import_rpc_data_from_file", "read", "load", "parse"):
        fn = getattr(reader, load_method, None)
        if callable(fn):
            fn()
            break

    data = None
    get_data = getattr(reader, "get_data", None)
    if callable(get_data):
        data = get_data()
    elif hasattr(reader, "data"):
        data = reader.data

    if data is None:
        raise RuntimeError("rpc_reader backend could not extract numeric channel data")

    channels = []
    get_channels = getattr(reader, "get_channels", None)
    if callable(get_channels):
        channels = get_channels() or []
    elif hasattr(reader, "channels"):
        channels = reader.channels or []

    headers = None
    get_headers = getattr(reader, "get_headers", None)
    if callable(get_headers):
        headers = get_headers()
    elif hasattr(reader, "headers"):
        headers = reader.headers

    return data, channels, _to_header_dict(headers)


def _load_via_pyrpc3(rsp_path: Path):
    from pyRPC3 import RPC3

    rpc = RPC3(str(rsp_path), debug=False)

    channels = getattr(rpc, "channels", []) or []
    if channels:
        series = []
        for ch in channels:
            values = (
                getattr(ch, "data", None)
                or getattr(ch, "values", None)
                or getattr(ch, "y", None)
            )
            values = list(values) if values is not None else []
            series.append(values)

        max_len = max((len(col) for col in series), default=0)
        padded = [col + [None] * (max_len - len(col)) for col in series]
        df = pd.DataFrame(padded).T
        df.columns = _extract_channel_names(channels, df.shape[1])

        headers = {
            "backend": "pyRPC3",
            "channel_count": len(channels),
            "row_count": len(df),
        }
        return df, headers

    matrix = getattr(rpc, "data", None)
    if matrix is None:
        raise RuntimeError("pyRPC3 backend did not expose channel data")

    df = pd.DataFrame(matrix)
    df.columns = _extract_channel_names([], df.shape[1])
    headers = {
        "backend": "pyRPC3",
        "channel_count": df.shape[1],
        "row_count": len(df),
    }
    return df, headers


def _extract_channel_units(channels, width):
    units = []
    for i in range(width):
        ch = channels[i] if i < len(channels) else None
        unit = None

        if isinstance(ch, dict):
            unit = ch.get("units") or ch.get("Units")
        elif ch is not None and not isinstance(ch, str):
            unit = getattr(ch, "units", None) or getattr(ch, "unit", None)

        units.append(str(unit).strip() if unit else "")
    return units


def _resolve_dt(header_dict):
    for key in (
        "DELTA_T",
        "DELTA T",
        "DELTAT",
        "delta_t",
        "sample_interval",
        "x_increment",
    ):
        if key in header_dict:
            try:
                return float(header_dict[key])
            except Exception:
                pass
    return 1.0


def _load_reference_titles_if_present(rsp_path: Path, expected_count: int):
    """If a known simultaneous-values export exists, reuse its channel titles."""
    candidates = [
        rsp_path.with_name(f"{rsp_path.stem}_simultaneous_values00 copy.csv"),
        rsp_path.with_name(f"{rsp_path.stem}_simultaneous_values00.csv"),
    ]

    for cand in candidates:
        if not cand.exists():
            continue
        try:
            with cand.open("r", encoding="utf-8", newline="") as f:
                rows = list(csv.reader(f))
            if len(rows) >= 5 and rows[3] and rows[3][0] == "#TITLES":
                titles = rows[4][7:]
                if len(titles) == expected_count:
                    return titles
        except Exception:
            continue

    return None


def rsp_to_tagged_csv(rsp_path, csv_path=None):
    rsp_path = Path(rsp_path)
    csv_path = Path(csv_path) if csv_path else rsp_path.with_suffix(".csv")

    backend_errors = []
    df = None
    header_dict = {}
    channels = []

    try:
        data, channels, headers = _load_via_rpc_reader(rsp_path)
        df = pd.DataFrame(data)
        df.columns = _extract_channel_names(channels, df.shape[1])
        header_dict = headers
        header_dict["backend"] = "rpc_reader"
        print("Backend: rpc_reader")
    except SystemExit as exc:
        # rpc_reader uses sys.exit() for validation errors (not a subclass of Exception).
        msg = exc.args[0] if exc.args else str(exc)
        backend_errors.append(f"rpc_reader failed: {msg}")
    except Exception as exc:
        backend_errors.append(f"rpc_reader failed: {exc}")

    if df is None:
        try:
            df, header_dict = _load_via_pyrpc3(rsp_path)
            channels = []
            header_dict["backend"] = "pyRPC3"
            print("Backend: pyRPC3")
        except SystemExit as exc:
            msg = exc.args[0] if exc.args else str(exc)
            backend_errors.append(f"pyRPC3 failed: {msg}")
        except Exception as exc:
            backend_errors.append(f"pyRPC3 failed: {exc}")

    if df is None:
        msg = "\n".join(backend_errors)
        raise RuntimeError(
            "No supported RSP parser worked. Install one of:\n"
            "  - python3 -m pip install rpc-reader\n"
            "  - python3 -m pip install git+https://github.com/galuszkm/pyRPC3.git\n"
            f"Details:\n{msg}"
        )

    channel_names = list(df.columns)
    titled_names = [f"{i + 1} {name}" for i, name in enumerate(channel_names)]

    # If parser did not provide real channel names, borrow them from a known
    # simultaneous-values reference export when available.
    if all(name.startswith("channel_") for name in channel_names):
        ref_titles = _load_reference_titles_if_present(rsp_path, len(channel_names))
        if ref_titles:
            titled_names = ref_titles

    units = _extract_channel_units(channels, len(channel_names))
    dt = _resolve_dt(header_dict)

    titles_prefix = [
        "Channel",
        "Feature Type",
        "Start Point",
        "End Point",
        "Start Time",
        "End Time",
        "Channel title",
    ]
    keywords_prefix = [
        "Channel",
        "Feature Type",
        "Start Point",
        "End Point",
        "Start Time",
        "End Time",
        "ChanTitle",
    ]
    datatypes_prefix = ["LONG", "LONG", "HUGE", "HUGE", "DOUBLE", "DOUBLE", "STRING"]

    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        # Match the tagged structure style of the reference file.
        writer.writerow(["#HEADER"])
        writer.writerow(["#CHANTITLE"])
        writer.writerow([header_dict.get("CHAN_TITLE", "Untitled")])
        writer.writerow(["#TITLES"])
        writer.writerow(titles_prefix + titled_names)
        writer.writerow(["#UNITS"])
        writer.writerow(["", "", "", "", "", "", ""] + units)
        writer.writerow(["#KEYWORDS"])
        writer.writerow(
            keywords_prefix + [f"Chan{i + 1}" for i in range(len(channel_names))]
        )
        writer.writerow(["#DATATYPES"])
        writer.writerow(datatypes_prefix + ["FLOAT"] * len(channel_names))
        writer.writerow(["#DATA"])

        # Simultaneous-values style output:
        # per channel, export feature 3 (minimum) and feature 4 (maximum),
        # and include all channel values at those time indices.
        for ch_idx in range(len(channel_names)):
            s = df.iloc[:, ch_idx]

            min_i = int(s.idxmin())
            min_point = min_i + 1
            min_t = round((min_point - 1) * dt, 12)
            min_values = df.iloc[min_i].tolist()
            writer.writerow(
                [
                    ch_idx + 1,
                    3,
                    min_point,
                    min_point,
                    min_t,
                    min_t,
                    titled_names[ch_idx] if ch_idx < len(titled_names) else "",
                ]
                + min_values
            )

            max_i = int(s.idxmax())
            max_point = max_i + 1
            max_t = round((max_point - 1) * dt, 12)
            max_values = df.iloc[max_i].tolist()
            writer.writerow(
                [
                    ch_idx + 1,
                    4,
                    max_point,
                    max_point,
                    max_t,
                    max_t,
                    titled_names[ch_idx] if ch_idx < len(titled_names) else "",
                ]
                + max_values
            )

    return csv_path, df.shape, len(channel_names), len(header_dict)


rsp_path = _find_repo_root() / "data/rsp_raw/4b3_k15543_L76_leaf_zr2_tkv114_Sep2021_t2xxf_lca_lf_app.rsp"
if not rsp_path.is_file():
    raise FileNotFoundError(
        f"RSP not found: {rsp_path}\n"
        f"cwd={Path.cwd()!s} repo_root={_find_repo_root()!s}"
    )
out, shape, channel_count, header_count = rsp_to_tagged_csv(rsp_path)
print(
    f"Converted: {rsp_path} -> {out} shape={shape} "
    f"channels={channel_count} headers_saved={header_count}"
)